In [3]:
import cv2
import numpy as np

def calculate_gate_distance(image_points, object_points, camera_matrix, dist_coeffs=None):
    """
    Calculates the relative distance and translation vector from the camera to the gate.
    """

    # Assume no lens distortion if coefficients aren't provided
    if dist_coeffs is None:
        dist_coeffs = np.zeros((4, 1))

    # SOLVEPNP_ITERATIVE is the default and works well for 4 coplanar points
    success, rotation_vector, translation_vector = cv2.solvePnP(
        object_points,
        image_points,
        camera_matrix,
        dist_coeffs,
        flags=cv2.SOLVEPNP_ITERATIVE
    ) #translation_vector --> (x,y,z) coordinates of gate centre assuming drone camera as origin
    # ie. Relative pose of the camera with respect to the gate

    if success:
        # The translation_vector contains the (X, Y, Z) distance in meters
        # from the camera to the center of the gate.
        # We calculate the Euclidean norm to get the straight-line distance.
        distance_to_gate = np.linalg.norm(translation_vector)
        return distance_to_gate, translation_vector
    else:
        return None, None

# ==========================================
# Example Usage:
# ==========================================

# Example 2D pixel outputs from your YOLO model [x, y] - yolo gate keypoints (coordinates)
yolo_keypoints = [
    [4, 5], # Top-Left
    [5, 6], # Top-Right
    [6, 7], # Bottom-Right
    [4, 7]  # Bottom-Left
]
image_points = np.array(yolo_keypoints, dtype=np.float32)

# Define the 3D coordinates of the gate in its LOCAL space.
# The A2RL gates have a 1.5m inner opening. We center the gate at (0, 0, 0).
half_size = 1.5 / 2.0 # assume gate centre as (0,0,0)
object_points = np.array([
    [-half_size,  half_size, 0.0], # Top-Left corner
    [ half_size,  half_size, 0.0], # Top-Right corner
    [ half_size, -half_size, 0.0], # Bottom-Right corner
    [-half_size, -half_size, 0.0]  # Bottom-Left corner
], dtype=np.float32)

# Example Camera Intrinsic Matrix (MUST calibrate on specific camera for accurate results)
# This matrix represents the focal length (fx, fy) and optical center (cx, cy)
camera_matrix = np.array([
    [800.0,   0.0, 320.0], # [fx,  0, cx]
    [  0.0, 800.0, 240.0], # [ 0, fy, cy]
    [  0.0,   0.0,   1.0]
], dtype=np.float32)

distance, tvec = calculate_gate_distance(image_points, object_points, camera_matrix)

if distance:
    print("tvec : ", tvec)
    print(f"Straight-line distance to gate: {distance:.2f} meters")
    print(f"Relative X (Left/Right): {tvec[0, 0]:.2f}m")
    print(f"Relative Y (Up/Down): {tvec[1, 0]:.2f}m")
    print(f"Relative Z (Forward): {tvec[2, 0]:.2f}m")

tvec :  [[-262.55906058]
 [-194.68135685]
 [ 666.28928048]]
Straight-line distance to gate: 742.15 meters
Relative X (Left/Right): -262.56m
Relative Y (Up/Down): -194.68m
Relative Z (Forward): 666.29m
